# 🗣️ 声纹识别与说话人日志：谁在说话？

> 从入门到精通 · 白话 + 公式拆解 + 可执行代码

**本章定位**：识别“谁在说话”是语音的第二大任务族（第一是 ASR“说了什么”）。

**你将学到**：
1. 任务定义：声纹确认/辨识、说话人日志；评测指标 EER/DCF；
2. 特征演化：GMM-UBM → i-vector → x-vector(TDNN) → ECAPA-TDNN；
3. 关键损失：三元组/GE2E/AAM-Softmax（公式逐符号）；
4. 打分：余弦相似度、PLDA；
5. 说话人日志：VAD → 分段 → embedding → 聚类(AHC) → 重排；
6. 工具与工程：pyannote、部署注意。

**与其它章节关系**：依赖 00（特征/MFCC）；被 04、05（增强）前置；
会议系统 = 语音日志 + ASR 的经典组合。

In [1]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.RandomState(0)
np.set_printoptions(precision=4, suppress=True)
print('numpy', np.__version__, '| scipy OK')

numpy 2.4.3 | scipy OK


## 🍵 白话开篇：声纹 = 声音的“指纹”

声纹识别听一段声音，回答**两个经典问题**：
- **确认（Verification）**：这段声音是张三吗？（1 对 1）——门禁/支付/解锁；
- **辨识（Identification）**：这段声音是谁？（1 对 N）——在 1000 人里找；
- **日志（Diarization）**：这段录音里**有几个人、各在什么时间段**说话（不关心说的是什么）——会议纪要/刑侦。

**为什么“听声音认人”可行**：每个人的声道长短、口腔咽喉形状、说话习惯不同 →
共振峰位置、语速、韵律都带“个人签名”。

**为什么难（对比人脸）**：
- 同一个人**每次说话都不同**（情绪/场合/麦克风/噪声）；
- 不同人**可能很像**（双胞胎兄弟声线几乎一致）；
- 别人可能**模仿**（挑战声纹要防伪造）。

**一句话**：声纹不是又一个 ID 密码，而是一个“概率判断”——所以它永远有“误拒/误纳”的权衡。

## 🔍 公式拆解 ①：EER——误拒与误纳的平衡点

$$\text{FAR}(\tau)=\frac{\#(\text{冒名者得分}>\tau)}{\#(\text{冒名者测试})},\qquad \text{FRR}(\tau)=\frac{\#(\text{本人得分}<\tau)}{\#(\text{本人测试})}$$
$$\text{EER}=\tau^*\ \text{使得}\ FAR(\tau^*)=FRR(\tau^*),\qquad \text{DCF}=\min_\tau\ (C_{FA}\,P_{target}\,FAR + C_{FR}\,(1-P_{target})\,FRR)$$

**逐符号解释**：
- 得分对比阈值 $\tau$：分数 > τ 判“是本人”，否则“不是”；
- FAR（误纳/False Alarm）：**冒名者**被放进来（安全漏洞）；FRR（误拒/False Reject）**本人**被挡门外（体验伤害）；
- EER：把 τ 调到 FAR=FRR 那一点——**系统“两个错误一样贵”时的水平**，越低越好；
- DCF：**加权**成本（$C_{FA}$ 放错人代价大，$C_{FR}$ 挡错人代价大，$P_{target}$ 本来就该出现在测试里的先验）——比 EER 更贴近业务。

**直觉（为什么不用准确率）**：正负样本严重不平衡（冒名者测试几十万次），准确率没意义；
EER/DCF 只依赖“两条错误曲线”，与应用无关、可跨系统比较。

**数字体验**：EER=1% 表示“误拒和误纳同为 1%”——安全与体验各让一步。

In [2]:
# EER 计算演示：本人分数 ~ N(0.8, 0.2²)，冒名者分数 ~ N(0.2, 0.2²)
rng6 = np.random.RandomState(1)
gen = rng6.normal(0.8, 0.2, 20000)      # 本人（genuine）分数
imp = rng6.normal(0.2, 0.2, 20000)      # 冒名者（impostor）分数
taus = np.linspace(0, 1, 401)
far = np.array([(imp > tau).mean() for tau in taus])
frr = np.array([(gen < tau).mean() for tau in taus])
i_eer = np.argmin(np.abs(far - frr))
print('EER ≈ %.2f%%（阈值 τ=%.2f 处 FAR=%.3f, FRR=%.3f）'
      % (100*(far[i_eer]+frr[i_eer])/2, taus[i_eer], far[i_eer], frr[i_eer]))
plt.figure(figsize=(7, 2.8))
plt.plot(np.linspace(0, 1, 401), np.round(np.linspace(0,1,401),2)*0+far, lw=0)
plt.hist(imp, bins=60, alpha=0.5, label='冒名者分数', color='#b45309', density=True)
plt.hist(gen, bins=60, alpha=0.5, label='本人分数', color='#1f4e79', density=True)
plt.axvline(taus[i_eer], color='k', ls='--', lw=1)
plt.xlabel('相似度分数'); plt.ylabel('密度')
plt.legend(fontsize=8); plt.title('两条分布越分得开，EER 越低（阈值在交点）', fontsize=10)
plt.tight_layout(); plt.show()
plt.figure(figsize=(7, 2.4))
plt.plot(taus, far, label='FAR(误纳)', lw=1.5)
plt.plot(taus, frr, label='FRR(误拒)', lw=1.5)
plt.axvline(taus[i_eer], color='k', ls='--', lw=1)
plt.xlabel('阈值 τ'); plt.ylabel('错误率')
plt.legend(fontsize=8); plt.title('阈值往右移：更严格 → FAR 降 FRR 升（交点=EER）', fontsize=10)
plt.tight_layout(); plt.show()
assert (imp > taus[i_eer]).mean() > 0.005   # 基本是 1~3% 级别
print('✓ EER 报告格式：EER=%.2f%% @ τ=%.2f——论文/产品都这样写' % (100*(far[i_eer]+frr[i_eer])/2, taus[i_eer]))

EER ≈ 6.70%（阈值 τ=0.50 处 FAR=0.066, FRR=0.068）


✓ EER 报告格式：EER=6.70% @ τ=0.50——论文/产品都这样写


C:\Users\24260\AppData\Local\Temp\ipykernel_508\481868802.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
C:\Users\24260\AppData\Local\Temp\ipykernel_508\481868802.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [3]:
# DET 曲线：误拒率 vs 误识率（两条错误一升一降，交叉点 = EER）
rng6 = np.random.RandomState(2)
same = 0.80 + 0.12*rng6.randn(2000)       # 同一人的相似度（应高）
diff = 0.35 + 0.15*rng6.randn(2000)       # 不同人的相似度（应低）
ths = np.linspace(0.2, 0.95, 200)
frr = np.mean(same[:, None] < ths[None, :], axis=0)    # 误拒：同人判不同
far = np.mean(diff[:, None] > ths[None, :], axis=0)    # 误识：异人被接受
d = np.abs(frr - far); i = int(np.argmin(d))
eer = (frr[i] + far[i])/2
print('阈值 %.2f 处 FAR = %.1f%% | FRR = %.1f%% | EER ≈ %.1f%%' % (ths[i], 100*far[i], 100*frr[i], 100*eer))
plt.figure(figsize=(6.5, 2.8))
plt.plot(far*100, frr*100, lw=2)
plt.plot([far[i]*100], [frr[i]*100], 'ro', ms=8, label='EER 点')
plt.plot([0, 100], [0, 100], 'k--', lw=0.8, alpha=0.4)
plt.xlabel('FAR (%)'); plt.ylabel('FRR (%)')
plt.title('DET 曲线：低阈值→误识多，高阈值→误拒多（EER 越小越好）', fontsize=10)
plt.legend(fontsize=8); plt.tight_layout(); plt.show()
assert eer < 0.2
print('✓ EER 是“两种错误一样多”时的错率——比单点准确率更能描述声纹系统')


C:\Users\24260\AppData\Local\Temp\ipykernel_508\581463519.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(fontsize=8); plt.tight_layout(); plt.show()


阈值 0.60 处 FAR = 4.8% | FRR = 5.1% | EER ≈ 5.0%
✓ EER 是“两种错误一样多”时的错率——比单点准确率更能描述声纹系统


## 🍵 白话讲透声音指纹（embedding）

现代声纹的套路：**把一段任意长度的声音压成一个固定维度的向量（如 192 维）**——
这就是 **embedding（声纹向量/说话人向量）**。

**要求（三性）**：
- **说话人可分**：不同人 → 向量距离远；
- **内容无关**：说“你好”还是“再见”，向量尽量不变（这是难点！）；
- **条件鲁棒**：噪声/信道/情绪不变（增强+数据增强解决）。

**两条技术路线**：
- **概率模型时代**：GMM-UBM → i-vector（2011，经典）；
- **深度学习时代**：x-vector（2018，TDNN）→ ECAPA-TDNN（2020，当前主流基线）——
  网络自己从 MFCC/Fbank 里“提炼”说话人身份表示。

**下游怎么用向量**：注册时存 N 个向量，识别时算新向量与库里的余弦相似度/PLDA 分数。

**一句话**：embedding = “把声音压缩成指纹”；声纹系统 = 指纹提取器 + 相似度打分器。

## 🔍 公式拆解 ②：余弦相似度——向量夹角的“像不像”

$$\cos(\theta)=\frac{e_1\cdot e_2}{\|e_1\|\,\|e_2\|}=\frac{\sum_i e_{1i}e_{2i}}{\sqrt{\sum_i e_{1i}^2}\sqrt{\sum_i e_{2i}^2}}$$

**逐符号解释**：
- $e_1,e_2$：两个说话人向量（192 维左右）；分子：逐元素乘再求和（内积）；
- 分母：两个向量的**模长**——把“音量（模长）”归一掉，只看**方向**；
- 结果 ∈ [-1, 1]：1 = 同方向（同一人多不同录音也不到 1），0 = 垂直（无关），负数 = 反向（罕见）；
- 工程里常**先 L2 归一化**再算内积 = 余弦（省除法）。

**为什么用余弦不用欧氏距离**：
- embedding 的“绝对大小”常含说话音量/信道信息（没意义）；方向更纯地反映身份；
- 余弦对向量整体缩放不变（音量无关且实现简单）；
- 已有大量训练目标直接优化余弦（AAM-Softmax 就是）。

**数字小例**：$e_1=(1,0),\ e_2=(0.6,0.8)$ → 余弦 = 0.6（60° 夹角）；相同向量 → 1。

**一句话**：余弦 = “只看方向比不像”——声纹打分的主力，PLDA 是它的“升级版”。

In [4]:
# 余弦打分演示：随机 128 维单位向量，本人对=同人不同录音（带噪声扰动）
def cos_sim(a, b):
    return a @ b/(np.linalg.norm(a)*np.linalg.norm(b))
rng6 = np.random.RandomState(2)
n_spk = 30; D = 128
speakers = rng6.randn(n_spk, D)
speakers /= np.linalg.norm(speakers, axis=1, keepdims=True)
gen_score, imp_score = [], []
for spk in speakers:
    for _ in range(50):
        r1 = spk + 0.07*rng6.randn(D); r1 /= np.linalg.norm(r1)      # 同人另一段
        r2 = spk + 0.07*rng6.randn(D); r2 /= np.linalg.norm(r2)
        gen_score.append(cos_sim(r1, r2))
        o = speakers[rng6.randint(n_spk)]                            # 随机他人
        imp_score.append(cos_sim(spk, o))
gen_score, imp_score = np.array(gen_score), np.array(imp_score)
print('本人对均值 = %.3f（σ=%.3f）| 冒名对均值 = %.3f（σ=%.3f）'
      % (gen_score.mean(), gen_score.std(), imp_score.mean(), imp_score.std()))
plt.figure(figsize=(7, 2.5))
plt.hist(gen_score, bins=40, alpha=0.6, label='本人对', color='#1f4e79', density=True)
plt.hist(imp_score, bins=40, alpha=0.6, label='冒名对', color='#b45309', density=True)
plt.axvline(0.4, color='k', ls='--', lw=1, label='常用阈值 0.4')
plt.xlabel('余弦相似度'); plt.ylabel('密度')
plt.legend(fontsize=8); plt.title('用小扰动(=反复录音)与随机他人(=冒名)构造真假对', fontsize=10)
plt.tight_layout(); plt.show()
assert gen_score.mean() > imp_score.mean()
print('✓ 阈值 0.4 处：冒名对几乎全拒（安全），本人对几乎全过（体验）——余弦 0.4 是常见起点')

本人对均值 = 0.616（σ=0.044）| 冒名对均值 = 0.035（σ=0.191）
✓ 阈值 0.4 处：冒名对几乎全拒（安全），本人对几乎全过（体验）——余弦 0.4 是常见起点


C:\Users\24260\AppData\Local\Temp\ipykernel_508\1836232647.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🍵 白话讲透 i-vector：一句“我是谁”的浓缩

深度学习之前，主流是 **GMM-UBM → i-vector**（2011，Dehak）：

**第一层：UBM（通用背景模型）**
- 拿**几万人的语音**训一个大 GMM（如 1024 个高斯）——代表“所有人的平均声学空间”；
- 每个人的语音都“对齐”到这些高斯上（每帧属于哪个高斯分量）；
- 把每个人**每帧统计量（均值/协方差）按高斯拼接** → 超向量（几万维）。

**第二层：总变异空间（因子分析）**
- 超向量维度爆炸，但**人之间的差异其实很少**（声道、习惯）→ 用低维子空间建模：
  $$\mathbf{m} = \mathbf{m}_{UBM} + \mathbf{T}\mathbf{w}$$
- $\mathbf{T}$：总变异矩阵（几万×几百）；$\mathbf{w}$：**i-vector**（几百维低维指纹）。

**一句话**：i-vector = “把每个人说的一堆话，压成一个几百维向量”——
细节（内容、停顿）被丢掉，留下“你是谁”的痕迹。

## 🔍 公式拆解 ③：i-vector 的核心方程

$$\mathbf{m}=\mathbf{m}_{UBM}+\mathbf{T}\mathbf{w},\qquad \phi(\mathbf{x})=\sum_t \gamma_t(\mathbf{x})\,[\mathbf{x}_t-\mathbf{m}_{UBM}]$$

**逐符号解释**：
- $\mathbf{m}$：某人的超向量（GMM 均值拼接）；$\mathbf{m}_{UBM}$：通用均值（所有人的平均）；
- $\mathbf{T}$：总变异矩阵（列 = 说话人差异方向）；$\mathbf{w}$：**i-vector**（低维系数，就是指纹）；
- $\gamma_t(\mathbf{x})$：第 t 帧特征对第 c 个高斯的**软隶属度**（在 UBM 下的后验）；
- $\phi$：零阶统计量——按隶属度把每帧特征“加权对齐”到高斯中心。

**推导直觉**：把“全体方差”分成说话人间（T 方向）和说话人内（残差）两块；
统计量 $\phi$ 告诉我们“每个人偏离平均有多少”——$\mathbf{w}$ 就是这些偏差在子空间上的坐标。

**后续为什么要 PLDA**：i-vector 还混着信道/录音差异 → 用 PLDA 把“说话人因子”与“信道因子”分开再打分。

**一句话**：超向量 = 全文摘要，$\mathbf{T}\mathbf{w}$ = 只用主干维度表达“摘要”，丢写信人的笔迹但留内容风格。

In [5]:
# i-vector 思想迷你演示：PCA 子空间里的“指纹”（用 PCA 模拟 T 提取说话人差异）
rng6 = np.random.RandomState(3)
n_spk, n_utt, n_gauss, D = 20, 8, 64, 30
spk_means = rng6.randn(n_spk, n_gauss)                  # 每个人有个“说话人均值”
uts = [spk_means[i] + 0.15*rng6.randn(n_utt, n_gauss) for i in range(n_spk)]
all_u = np.vstack(uts)
mean_all = all_u.mean(axis=0)
U, S, Vt = np.linalg.svd(all_u - mean_all, full_matrices=False)
T = Vt[:5].T                                         # 低维“总变异”方向（特征空间 64→5）
iv = (all_u - mean_all) @ T                          # 每个人 → 5 维 i-vector
iv_spk = iv.reshape(n_spk, n_utt, 5).mean(axis=1)
within = ((iv - iv.reshape(n_spk, n_utt, 5).mean(axis=1).repeat(n_utt, axis=0))**2).sum(axis=1).mean()**0.5
between = np.std(iv_spk, axis=0).sum()/np.sqrt(5)
print('说话人内波动 = %.3f | 说话人间离散 = %.3f | 比值 = %.1f（越大越好分）'
      % (within, between, between/np.maximum(within, 1e-9)))
assert between > within
print('✓ PCA 视角：i-vector 就是“把高维差异投影到少数方向”——内容/噪声被丢弃，身份留下')

说话人内波动 = 0.310 | 说话人间离散 = 5.221 | 比值 = 16.8（越大越好分）
✓ PCA 视角：i-vector 就是“把高维差异投影到少数方向”——内容/噪声被丢弃，身份留下


## 🍵 白话讲透 PLDA：把“你是谁”和“在哪录的”分开

i-vector 里除了“说话人差异”，还混着**信道差异**（手机 vs 座机、不同麦克风）。
PLDA（概率线性判别分析）假设向量由**两个因子**组成：

$$\mathbf{x} = \boldsymbol{\mu} + \mathbf{F}\mathbf{h} + \mathbf{G}\mathbf{w} + \boldsymbol{\epsilon}$$

- $\mathbf{h}$：**说话人因子**（同一人的向量共享它——这就是“身份”）；
- $\mathbf{w}$：**信道/会话因子**（每次录音不同——噪声、设备、情绪）；
- 打分时比的是“两个向量是否共享同一个 $\mathbf{h}$”（似然比）。

**直觉**：两个人各说一段话，即使录音设备不同，只要“说话人因子”相同就判本人——
PLDA 在 i-vector 时代把 EER 压低了 30%~50%。

**深度学习时代还要 PLDA 吗**：x-vector/ECAPA 已经更“干净”，但**加了更好**——
用 PLDA 或余弦+长度归一化（length-norm）做后端打分仍是标配。

**一句话**：PLDA = “把向量拆成身份部分和环境部分，再只比身份部分”。

## 🔍 公式拆解 ⑤：PLDA 打分（似然比）

$$s(x_1,x_2) = \log \frac{p(x_1,x_2\mid H_1)}{p(x_1,x_2\mid H_0)}$$
$$H_1:\ \text{同一个人} \qquad H_0:\ \text{不同的人}$$

**逐符号解释**：
- $H_1$（同人假设）：两个向量来自同一个说话人因子 $\mathbf{h}$；$H_0$：分别来自不同 $\mathbf{h}$；
- $p(x_1,x_2\mid H_1)$：同人假设下“出现这对向量”的概率（说话人因子匹配度高 → 大）；
- 比值（似然比）取对数：>0 支持同人，<0 支持不同人；
- 参数 $\mathbf{F},\mathbf{G},\boldsymbol{\mu}$ 用训练集 EM/矩估计学出来（谁与谁同源未知，靠“随机对”约束）。

**与余弦的关系**：余弦只看“夹角”，不显式建模信道；PLDA 是“生成式”打分——
数据多时通常更准；但要求向量分布近高斯，所以要先做**长度归一化**。

**一句话**：PLDA 分数 = “这两段话更像同一个人还是不同人”的 log 似然比——
声纹后端的“正规军”。

## 🍵 白话讲透说话人日志：一场圆桌谁说了几句

**任务**：输入一整段多人录音（会议/访谈），输出：
```
[0.0s-3.2s] 说话人A     [3.2s-8.1s] 说话人B
[8.1s-12.5s] 说话人A   [12.5s-14.0s] 沉默
```

**标准流水线（四步，背下来）**：
1. **VAD**：切出“有人说话”的片段（丢掉静音/噪声）；
2. **分段（segmentation）**：把语音切成 1~3 秒小段（说话人可能在切换）；
3. **提取 embedding**：每段 → 一个声纹向量（ECAPA）；
4. **聚类**：按相似度把段分给不同说话人（AHC/谱聚类），未知人数自动定。

**难点**：
- 人数**未知**（聚类要自己猜）；
- 说话人**中途换位置/换麦**（embedding 漂移）；
- 重叠说话（两人同时开口）——新方向是**端到端**（EEND，序列标注直接出日志）。

**评测**：DER（Diarization Error Rate）= 误说话人时间 + 漏检 + 误检。

**一句话**：日志 = “VAD + 声纹 + 聚类”的流水线；产出对齐时间线，供 ASR 逐人转写。

In [6]:
# 说话人日志最后一步：AHC（自底向上凝聚聚类）——每次合并最相似的两簇
rng6 = np.random.RandomState(4)
n_seg, D = 24, 32
centers = np.array([[0., 0.], [3., 0.], [0., 3.]])      # 3 个说话人的中心
lab_true = rng6.randint(0, 3, n_seg)
segs = centers[lab_true] + 0.25*rng6.randn(n_seg, 2)     # 简化成 2 维便于看
def ahc(X, tol):
    clusters = [[i] for i in range(len(X))]
    C = [np.array(X[i], dtype=float) for i in range(len(X))]
    while len(C) > 1:
        d = np.array([[np.linalg.norm(C[i]-C[j]) for j in range(len(C))] for i in range(len(C))])
        np.fill_diagonal(d, np.inf)
        i, j = np.unravel_index(np.argmin(d), d.shape)
        if d[i, j] > tol:
            break
        li, lj = len(clusters[i]), len(clusters[j])          # 按簇大小加权平均合并
        clusters[i] += clusters[j]; del clusters[j]
        C[i] = (li*C[i] + lj*C[j])/(li+lj); del C[j]
    return clusters, C
clusters, _ = ahc(segs, tol=1.2)
lab_pred = np.empty(n_seg, dtype=int)
for ci, cl in enumerate(clusters):
    for s in cl:
        lab_pred[s] = ci
from itertools import permutations
acc = max(np.mean([int(perm[lab_pred[s]] == lab_true[s]) for s in range(n_seg)])
          for perm in permutations(range(3)))               # 编号可置换，取最优对应
print('AHC 自动发现 %d 个说话人 | 最优编号对应后一致率 = %.0f%%' % (len(clusters), 100*acc))
plt.figure(figsize=(7, 2.6))
plt.scatter(segs[:, 0], segs[:, 1], c=lab_pred, cmap='Set1', s=40)
for s in range(len(segs)):
    plt.text(segs[s, 0]+0.07, segs[s, 1]+0.07, 's%d' % s, fontsize=7)
plt.title('AHC 结果：颜色=自动聚类，s 编号=原始段（应聚成 3 团）', fontsize=10)
plt.xlabel('维1'); plt.ylabel('维2'); plt.tight_layout(); plt.show()
assert len(clusters) == 3 and acc > 0.9
print('✓ 聚类数未知也能跑：合并到“最远距离超阈值”就停——工程上再按 DER 调阈值')

AHC 自动发现 3 个说话人 | 最优编号对应后一致率 = 100%
✓ 聚类数未知也能跑：合并到“最远距离超阈值”就停——工程上再按 DER 调阈值


C:\Users\24260\AppData\Local\Temp\ipykernel_508\2646812481.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel('维1'); plt.ylabel('维2'); plt.tight_layout(); plt.show()


## 🍵 白话讲透端到端日志（EEND）：让网络直接排时间线

传统四步流水线有**误差累积**（VAD 错→分段错→聚类错）。
**端到端（EEND）**把“音频 → 每人每时刻说话与否”当**序列标注**直接学：

- 输入：整段特征；输出：$[0/1]^{T \times K}$（K 个假设说话人，每帧谁在说）；
- 自注意力/Transformer 建模长程（谁在几秒后回头接话）；
- 损失用 PIT（置换不变训练）——因为“说话人 1/2”只是编号，谁是谁无所谓，
  取所有编号排列里损失最小的那个。

**现状**：EEND 在小人数（2~4）效果已接近流水线；重叠语音天然支持；
大型会议仍常“流水线 + EEND 后处理”混用。

**一句话**：EEND = “别分步，直接学输出时间线”——PIT 解决“编号无意义”的问题。

## 🔍 公式拆解 ⑥：PIT——聚类结果怎么算损失

$$L = \min_{\pi\in S_K}\ \frac1K\sum_{k=1}^{K} \text{BCE}(\hat y_{\pi(k)}, y_k)$$

**逐符号解释**：
- $y_k$：真实“第 k 人是否在说话”序列；$\hat y_k$：网络预测；
- $S_K$：K 个编号的**所有排列**（K=4 时 24 种）；$\pi$：一种编号对应方案；
- 对每种对应方案算平均 BCE，取**最小**——网络不用管“网络输出 1 号到底该对应谁”。

**为什么必须 PIT**：说话人标签是**任意编号**的（把 A 叫 1 或 2 都行）——
没有 PIT 网络会被“编号不一致”的梯度搞糊涂。

**代价**：K 增大排列爆炸（K=10 → 362 万）→ 用匈牙利匹配/贪心近似。

**一句话**：PIT = “反正编号无所谓，取最像的那种对法来算错”。

In [7]:
# PIT 数值演示：预测输出顺序乱了，损失仍能算对（取排列最小值）
rng6 = np.random.RandomState(5)
K = 3; T = 100
y_true = (rng6.rand(K, T) > 0.6).astype(float)
y_pred = np.vstack([y_true[1], y_true[2], y_true[0]]) + 0.1*rng6.randn(K, T)  # 编号错位
def bce(a, b):
    p = np.clip(a, 1e-6, 1-1e-6)
    return -(b*np.log(p) + (1-b)*np.log(1-p)).mean()
import itertools
best = min((bce(y_pred[list(perm)], y_true).mean() for perm in itertools.permutations(range(K))))
naive = bce(y_pred, y_true).mean()
print('逐位直接算损失 = %.3f | PIT(取最优排列) = %.3f（更小=算对了）' % (naive, best))
assert best < naive
print('✓ PIT 本质：先把“谁对应谁”配好，再报损失——避免编号错位惩罚模型')

逐位直接算损失 = 3.924 | PIT(取最优排列) = 0.039（更小=算对了）
✓ PIT 本质：先把“谁对应谁”配好，再报损失——避免编号错位惩罚模型


## 🍵 白话讲透 x-vector：让网络自己提炼指纹

i-vector 的问题是：统计量设计**手搓**（对齐到哪个高斯、怎么拼）——
深度学习版本 **x-vector**（TDNN，Snyder 2018）直接学：

**结构（TDNN + 统计池化 + 分类头）**：
```
帧级特征(Fbank) → [TDNNx5 卷积(帧上下文)] → 统计池化(帧均值+标准差)
  → [全连接段级层] → 说话人分类(softmax N 类)
                    → 倒数第二层输出 = x-vector (512维)
```

**三个关键设计（背下来）**：
1. **TDNN（时延网络）**：每层看“±几帧”的局部上下文 → 从短时特征里逐步挑出音色线索；
2. **统计池化**：把“变帧数”变成“定长”（均值+标准差）→ 任意时长录音都能出一个向量；
3. **分类头**：训练时按说话人分类，**训练完把分类头丢掉**，用池化后的向量当指纹——
   “为了分类学到的表示，迁移来做相似度”。

**为什么比 i-vector 强**：端到端优化“区分说话人”目标本身，不做中间统计量假设；
数据够多时上限更高。

**一句话**：x-vector = “卷积提特征 + 池化定长 + 分类学表示”三件套——声纹的 ResNet 时刻。

## 🔍 公式拆解 ④：AAM-Softmax——把向量“按角度拉远”

$$L = -\log\frac{e^{s\cos(\theta_{y}+m)}}{e^{s\cos(\theta_y+m)}+\sum_{j\ne y}e^{s\cos\theta_j}}$$

**逐符号解释**：
- $\theta_y$：样本向量与其**真类**中心的夹角；$\theta_j$：与其它类中心的夹角；
- $s$：缩放因子（把分数放大到 30~64，梯度更舒服）；
- $m$：**角度间距**（如 0.2~0.5 rad）——把真类要求“更苛刻”：
  不仅要分数最高，还要**角度额外小 m**；
- 效果：类中心被“推开”，同一说话人的向量聚得更紧 → 余弦打分更可靠。

**为什么 AAM（Additive Angular Margin）流行**：
- 直接在**余弦空间**加约束，与推理时的余弦打分**一致**（训练推理同域）；
- 比三元组（构造样本对难、收敛慢）稳定得多——工业界默认；
- 人脸识别（ArcFace）与声纹（ECAPA 系）都在用。

**一句话**：AAM-Softmax = “分类训练 + 要求真类分数再高一点”——
训练时就为“分开不同人”预留余量。

In [8]:
# AAM-Softmax 前向数值：其他条件相同，加 margin m 后损失更大（逼模型更努力）
def aam_loss(cos_theta_y, cos_others, s=30.0, m=0.3):
    theta_y = np.arccos(np.clip(cos_theta_y, -1, 1))
    num = np.exp(s*np.cos(theta_y + m))
    den = num + np.sum(np.exp(s*cos_others))
    return -np.log(num/den)
cos_y = 0.85; cos_others = np.array([0.3, -0.2, 0.1])
l0 = aam_loss(cos_y, cos_others, m=0.0)
l1 = aam_loss(cos_y, cos_others, m=0.3)
print('无 margin 损失 = %.4f | 加 margin=0.3 损失 = %.4f（↑ %.3f）' % (l0, l1, l1-l0))
ms = np.linspace(0, 0.6, 7)
losses = [aam_loss(cos_y, cos_others, m=mm) for mm in ms]
plt.figure(figsize=(7, 2.4))
plt.plot(ms, losses, 'o-')
plt.xlabel('角度间距 m (rad)'); plt.ylabel('损失')
plt.title('AAM：m 越大损失越大 → 优化器更早“把向量拉向类中心”', fontsize=10)
plt.tight_layout(); plt.show()
assert l1 > l0
print('✓ m 的代价：训练更难收敛（损失更大）；收益：推理余弦打分更“两极分化”')

无 margin 损失 = 0.0000 | 加 margin=0.3 损失 = 0.0000（↑ 0.000）


✓ m 的代价：训练更难收敛（损失更大）；收益：推理余弦打分更“两极分化”


C:\Users\24260\AppData\Local\Temp\ipykernel_508\2249860652.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 🍵 白话讲透 ECAPA-TDNN：当前声纹基线

ECAPA-TDNN（2020，VoxCeleb 冠军基线）在 x-vector 上做了**三处升级**：

1. **SE-Res2Net 块**：每个卷积块加**通道注意力**（SENet 那套）+
   Res2Net 多尺度——让网络知道“哪个频带对身份更重要”（如共振峰集中区）；
2. **多尺度特征融合**：第 1/2/3 层的特征都取出来相加再过池化——低层细节+高层语义都用上；
3. **加宽统计池化**：算均值+标准差还不够，再加**二阶统计**（或更大池化特征）→
   把“说话人特性”榨得更干。

**配合 AAM-Softmax + 更强数据增强**（SpecAugment/加噪/加混响/变声速），
在 VoxCeleb1 上 EER 从 x-vector 的 ~4% 降到 ~0.8%（远距离说话人验证）。

**工程意义**：ECAPA 系列是开箱即用的标准 baseline（SpeechBrain/WeSpeaker 都有），
面试说“声纹用 ECAPA-TDNN + AAM-Softmax”即主流回答。

**一句话**：ECAPA = x-vector + 通道注意力 + 多尺度 + 更厚池化——精装修版指纹提取器。

## 🛡️ embedding 鲁棒性：训练数据增强清单

声纹系统换个麦克风/房间就“不认识”了？——训练时要让 embedding 对不同条件不敏感：

- **说话人扰动**：语速 ±10%、音高 ±2 半音、音量随机；
- **信道模拟**：随机 EQ/压缩/MOS 噪声（模拟不同手机）；
- **混响**：随机 RIR 卷积（大/小房间、远近麦克风）；
- **SpecAugment**：随机遮挡频带（逼模型别只盯一个共振峰）；
- **远端/近端切换**：训练里混入远场语料。

**变长语音处理**：句子长短不一 → 均值池化（mean pooling）或注意力池化；
**归一化**：embedding 建议 L2 归一化（只比方向不比幅度，cosine 一致）。

**一句话**：声纹模型 90% 的“翻车”是训练条件单一造成的——数据增强是刚需。


## 🗂️ 数据与工具（面试要能点名）

**经典数据集**：
- **VoxCeleb1/2**：YouTube 名人语音，1k/6k+ 说话人——声纹训练标配；
- **LibriSpeech**（语音书，可用于说话人）；**CN-CELEB**（中文名人声纹基准）；
- **AMI 会议**（日志 benchmark）；**DIHARD**（日志挑战赛，难场景）；
- 英文评测协议：VoxCeleb1-O（远距离验证对）。

**开源工具（现成可用）**：
- **SpeechBrain**：x-vector/ECAPA 一键训练 + 推理；
- **WeSpeaker**（阿里）：中文友好、ECAPA 基线；
- **pyannote.audio**：日志全家桶（VAD+分割+聚类，预训练模型）；
- **Kaldi**：i-vector/PLDA 经典实现（研究仍可参考）。

**一句话**：说得出 VoxCeleb/pyannote/ECAPA+SpeechBrain，面试官就知道你真上手过。

## 🚨 防伪（anti-spoofing）：声纹系统必须防的“三把刀”

声纹再准也怕假声音。三类攻击（背下来）：

1. **重放攻击**：录音直接放——麦克风/扬声器响度、频谱细节可查（重放无真实录音的“呼吸纹理”）；
2. **合成攻击**：TTS/VC 伪造（03/07 章技术被反用）——检测声学细节不自然处；
3. **对抗扰动**：在波形上加人耳听不见的噪声，骗过声纹打分——需对抗训练。

**评测基准**：ASVspoof（LA 逻辑访问 / DF 深度伪造）——标准数据 + EER 指标；
**常见 countermeasure（CM）**：后端特征（LFCC/FFT 残差）+ 轻量分类器；
**趋势**：大模型合成语音越来越像 → 水印（AudioSeal）+ 来源声明是部署必须。

**一句话**：声纹上线 = 识别 + 防伪双引擎；只做识别不做防伪 = 给攻击者留门。


## ⚠️ 本章高频踩坑

1. **内容泄漏**：测试对与训练对用了同一句话 → 模型作弊（要说话人/内容双分留出）；
2. **固定阈值走天下**：不同噪声/信道最优阈值不同——按场景调（或自适应）；
3. **短语音硬上**：<1s 的语音 embedding 不稳——先做“积累 3~5s”再打分；
4. **只用余弦**：数据多时 PLDA/长度归一化还能再降 EER；
5. **EER 当唯一指标**：业务要 DCF（安全场景误纳代价高，阈值要往保守调）；
6. **日志聚类忽略 VAD 质量**：VAD 漏检直接变成日志漏检——先修 VAD；
7. **未知人数硬定 K**：聚类要先自动定 K（用距离阈值/信息准则）；
8. **防伪（anti-spoofing）缺失**：录音回放攻击——生产要加活体检测/深度伪造检测。

## ⚙️ 声纹系统的部署工程注意

- **端侧 vs 云侧**：注册/验证放端侧（隐私）+ 云端做大库检索（1:N）；
- **量化**：x-vector/ECAPA 可 int8 量化，端侧功耗下降、精度损失可控（先测 EER 差）；
- **模板更新**：长期用户声音渐变 → 定期用“增量注册”更新 enrollment（别让老模板失效）；
- **多语言/口音**：embedding 对语种不敏感是目标，但注册样本要覆盖口音；
- **隐私合规**：声音属敏感生物信息——模板不可逆存储（加盐哈希化 embedding）、可删除。

**一句话**：声纹的坑大多在“工程上线”而非“模型指标”——面试聊到部署细节很加分。


## 🎤 面试 8 连追问

**Q1：声纹确认 vs 辨识 vs 日志？**
→ 1对1 验证、1对N 检索、无标注“谁何时说话”的分割。

**Q2：EER 怎么得到？阈值往哪移是更严格？**
→ 扫描阈值找 FAR=FRR 交点；阈值调高 = 更严格 = FAR↓FRR↑。

**Q3：为什么需要 embedding 而不是直接比波形？**
→ 波形含内容/噪声/信道信息；embedding 压缩成身份表示，可跨时长打分。

**Q4：i-vector 和 x-vector 本质区别？**
→ i-vector 是 GMM 统计量+因子分析的浅层模型；x-vector 是网络端到端学表示。

**Q5：AAM-Softmax 为什么比三元组常用？**
→ 训练稳定（无难样本挖掘）、在余弦空间直接加 margin、与推理打分一致。

**Q6：PLDA 解决 i-vector 的什么问题？**
→ 把说话人因子和信道因子分开，只比身份部分。

**Q7：日志流水线四步？误差怎么传播？**
→ VAD→分段→embedding→聚类；每步错都往下传，所以有人做端到端。

**Q8：PIT 解决什么？**
→ 说话人编号任意，取最优排列算损失（EEND 的训练关键）。

## ✅ 10 题自测

1. 写 FAR/FRR 公式并说明 EER 定义；
2. 余弦相似度公式与为什么 L2 归一后就是余弦；
3. i-vector 方程 $\mathbf{m}=\mathbf{m}_{UBM}+\mathbf{T}\mathbf{w}$ 每个符号；
4. x-vector 的三段结构与统计池化作用；
5. AAM-Softmax 公式、m 与 s 的含义；
6. PLDA 打分为什么是似然比；
7. 日志四步流水线；AHC 怎么定 K；
8. PIT 公式与为什么需要；
9. EER 与 DCF 的适用场景区别；
10. 说出 2 个数据集 + 2 个工具。
（答案在对应小节，能默写即过关）

## 🗺️ 声纹在语音全链路中的位置

```
增强(05) → 特征(00) → [本章: 声纹embedding] → 余弦/PLDA打分 → 确认/辨识
                          ↓
         VAD → 分段 → embedding → AHC聚类 → 说话人日志(会议)
                          ↓
             日志+ASR 联合：逐人转写（会议纪要/客服质检）
```

- 与 **ASR**：日志给 ASR 提供“谁的语音”，ASR 给日志提供“什么时候在说”（VAD 借力）；
- 与 **增强**：干净输入 → embedding 更稳 → EER 降；
- 与 **TTS/转换**：音色建模是它们的核心组件（下一章 07 大量使用）。

**下一章预告**：把“谁在说话”反过来用——把 A 的音色换成 B（07-语音转换与歌声转换）。

## 🏁 本章学完清单

- [ ] 能区分确认/辨识/日志并各举一例；
- [ ] 能手算/画图理解 EER 与 DCF；
- [ ] 能写余弦相似度并解释方向意义；
- [ ] 能解释 i-vector 的 UBM+总变异思想；
- [ ] 能画出 x-vector/ECAPA 的结构三段；
- [ ] 能写 AAM-Softmax 公式并解释 margin；
- [ ] 能解释 PLDA 似然比打分；
- [ ] 能默写日志四步流水线与 AHC 流程；
- [ ] 能写 PIT 公式并说明用途；
- [ ] 能点名 VoxCeleb/CN-CELEB/DIHARD/pyannote/SpeechBrain。

> 完成！下一本：07-语音转换与歌声转换 SVC。